In [ ]:
!pip install librosa torch torchvision pandas matplotlib scikit-learn seaborn


In [ ]:

import json
import os
import zipfile

# REPLACE THESE WITH YOUR DETAILS
# Your username is likely in the top right of the Kaggle page
username = "amoghlalwani"
# The token you just copied
key = "KGAT_11946681fc2f6f055f3bf983f1a6712c"

# Create the json file programmatically
data = {"username": username, "key": key}
with open('kaggle.json', 'w') as f:
    json.dump(data, f)

# Move it to the right place
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Test download
!kaggle datasets download uwrfkaggler/ravdess-emotional-speech-audio

zip_path = '/content/ravdess-emotional-speech-audio.zip'
extract_path = '/content/RAVDESS'

# Unzip the file
print("Unzipping data...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"SUCCESS: Data extracted to {extract_path}")

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
CONFIG = {
    'SR': 22050,
    'DURATION': 3.0,
    'N_MELS': 128,
    'HOP_LEN': 512,
    'BATCH_SIZE': 64,    # CHANGED: Increased from 16 to 64 for speed
    'EPOCHS': 75,        # CHANGED: Increased to 40 since it will run fast now
    'LR': 0.00005,
    'DATA_PATH': '/content/RAVDESS'
}

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
from tqdm import tqdm

class RAVDESSDataset(Dataset):
    def __init__(self, root_dir, transform=None, is_train=False):
        self.root_dir = root_dir
        self.is_train = is_train
        self.data = []
        self.labels = []

        print("Pre-loading dataset... (Robust Version)")

        valid_files = []
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.endswith('.wav'):
                    try:
                        if len(file.split('-')) == 7:
                            valid_files.append(os.path.join(root, file))
                    except:
                        continue

        # Process files
        for path in tqdm(valid_files):
            try:
                # 1. Parse Label
                parts = os.path.basename(path).split('-')
                emotion = int(parts[2]) - 1

                # 2. Load Audio
                y, sr = librosa.load(path, sr=CONFIG['SR'])

                # 3. Trim Silence (More aggressive trim)
                y, _ = librosa.effects.trim(y, top_db=30)

                # 4. Pad/Truncate to target length
                target_len = int(CONFIG['SR'] * CONFIG['DURATION'])
                if len(y) > target_len:
                    y = y[:target_len]
                else:
                    y = np.pad(y, (0, target_len - len(y)), mode='constant')

                # 5. Spectrogram
                mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=CONFIG['N_MELS'], hop_length=CONFIG['HOP_LEN'])
                mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

                # 6. Robust Normalization (Mean/Std instead of Min/Max)
                mean = np.mean(mel_spec_db)
                std = np.std(mel_spec_db)
                mel_spec_norm = (mel_spec_db - mean) / (std + 1e-6)

                self.data.append(torch.tensor(mel_spec_norm, dtype=torch.float32).unsqueeze(0))
                self.labels.append(emotion)

            except Exception as e:
                continue

        self.labels = torch.tensor(self.labels, dtype=torch.long)
        print(f"Loaded {len(self.data)} samples.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [ ]:
# Initialize full dataset
full_dataset = RAVDESSDataset(CONFIG['DATA_PATH'], is_train=False)

# Calculate split sizes
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

# Split
train_data, val_data, test_data = random_split(full_dataset, [train_size, val_size, test_size])

# Mark training set as "is_train=True" to enable augmentation
train_data.dataset.is_train = True

# Create Loaders
train_loader = DataLoader(train_data, batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
val_loader = DataLoader(val_data, batch_size=CONFIG['BATCH_SIZE'], shuffle=False)
test_loader = DataLoader(test_data, batch_size=CONFIG['BATCH_SIZE'], shuffle=False)

print(f"Data Split -> Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

In [ ]:
class SER_CNN(nn.Module):
    def __init__(self):
        super(SER_CNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)

        # Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Global Pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # Classifier
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 8) # 8 Emotions

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.view(x.size(0), -1) # Flatten

        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        output = self.fc2(x)
        return output

model = SER_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['LR'])

In [ ]:
def train_model(model, train_loader, val_loader, epochs):
    history = {'train_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f} - Val Acc: {val_acc:.2f}%")
        history['train_loss'].append(running_loss/len(train_loader))
        history['val_acc'].append(val_acc)

    return history

# RUN TRAINING
history = train_model(model, train_loader, val_loader, CONFIG['EPOCHS'])

# Save the model [cite: 250]
torch.save(model.state_dict(), 'ser_model_weights.pth')
print("Model Saved!")

In [ ]:
# 1. Generate Confusion Matrix & F1 Score
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Macro F1 Score
f1 = f1_score(all_labels, all_preds, average='macro')
print(f"Macro F1-Score: {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
emotions = ['Neutral', 'Calm', 'Happy', 'Sad', 'Angry', 'Fearful', 'Disgust', 'Surprised']

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=emotions, yticklabels=emotions, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# 2. Gender Bias Check [cite: 247]
# (This logic is simplified; for a rigorous check, split the test set by gender ID from filenames)
print("Note: To strictly test Pitch Bias, re-run evaluation on Male vs Female subsets of the test data.")